# 🎯 [DAT06_M09] AAR 기반 결산 수치 오집계 사례 분석 및 집계 룰셋/템플릿 보정
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m09_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M09 |
| **미션 주제** | AAR 기반 결산 수치 오집계 사례 분석 및 집계 룰셋/템플릿 보정 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 결산 수치 오집계 사례 분석 및 집계 룰셋/템플릿 보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 결산 수치 오집계 사례를 분석하고 룰셋을 보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **AAR 사례의 결산 수치 오집계 5유형을 재현·분석하고 집계 룰셋과 템플릿을 보정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **중복 제출, 단위 혼용(kL), 정정 문서 누락, 박스·발 혼용, 품목 별칭 누락의 5유형을 가상 제출 자료에 심어 룰셋 v1의 오집계를 재현 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/dat06_uat_feedback_report.md 및 패치 내역` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 실무 업무 생산성 기여도 및 현장 적응성

#### 미션 PASS 조건

- 정량 기준: UAT 체크리스트 만족도 90점 이상, 개선 요구 5건 반영 완료


- 진행 기준: UAT 결과 보고서 작성 및 최종 버그 픽스 완료


- 검증 방법: 실제 업무 시나리오 기반 최종 검수 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""사후검토(AAR) 사례에서 결산 수치 오집계 5유형(중복 행, 단위 혼용, 정정 누락, 박스·발 혼용, 품목 별칭 누락)을 재현해 원인을 분석하고 집계 룰셋 v1을 v2로 보정한 뒤, 재집계 결과와 UAT 체크리스트 점수, 개선 요구 반영 현황, 패치 내역을 출력한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

# 1) 가상 제출 레코드 생성: 정답(순불출 = 불출 - 반납)과 부대가 실제로 낸 제출 형태(오집계 유발 요인 포함)
led = pd.DataFrame(make_ledger(1))
rng = np.random.default_rng(9)
truth = {(r.unit, r.item): r.issue - r.ret for r in led.itertuples()}
subs, doc = [], 0
def add(u, it, kind, qty, uom, name=None):
    global doc
    doc += 1
    subs.append({"doc": f"D{doc:05d}", "unit": u, "item": name or it, "kind": kind, "qty": qty, "uom": uom})
for r in led.itertuples():
    add(r.unit, r.item, "issue", r.issue, UNIT_CODE[r.item]); add(r.unit, r.item, "return", r.ret, UNIT_CODE[r.item])
keys = list(truth)
pick = lambda n, f=lambda k: True: [keys[i] for i in rng.choice([j for j, k in enumerate(keys) if f(k)], n, replace=False)]
CASE = {}
for k in pick(6):                                   # 사례1: 같은 문서번호가 중복 제출
    CASE.setdefault("중복 행", []).append(k); subs.append(dict(next(s for s in subs if (s["unit"], s["item"]) == k and s["kind"] == "issue")))
for k in pick(4, lambda k: k[1] == "유류"):         # 사례2: 유류를 kL 단위로 제출(1kL = 1000L)
    CASE.setdefault("단위 혼용(kL)", []).append(k)
    for s in subs:
        if (s["unit"], s["item"]) == k:
            s["qty"], s["uom"] = s["qty"] / 1000, "kL"
for k in pick(5):                                   # 사례3: 불출 후 정정 문서(음수 조정)를 별도로 제출
    CASE.setdefault("정정 누락", []).append(k); d = int(rng.integers(5, 20))
    next(s for s in subs if (s["unit"], s["item"]) == k and s["kind"] == "issue")["qty"] += d
    add(k[0], k[1], "adjust", -d, UNIT_CODE[k[1]])
for k in pick(4, lambda k: k[1].startswith("훈련탄")):   # 사례4: 탄약을 박스(BX, 1BX=20발)로 제출
    CASE.setdefault("박스·발 혼용", []).append(k)
    for s in subs:
        if (s["unit"], s["item"]) == k:
            s["qty"], s["uom"] = s["qty"] / 20, "BX"
for k in pick(5, lambda k: k[1] == "피복류"):        # 사례5: 품목명을 줄여서 제출('피복')
    CASE.setdefault("품목 별칭 누락", []).append(k)
    for s in subs:
        if (s["unit"], s["item"]) == k:
            s["item"] = "피복"
S = pd.DataFrame(subs)

# 2) 집계 룰셋 v1(AAR 이전) / v2(보정 후)
def aggregate(S, ruleset):
    d = S.copy()
    if ruleset == "v2":
        d = d.drop_duplicates("doc")
        d["item"] = d["item"].replace({"피복": "피복류"})
        d = d[d["item"].isin(UNIT_CODE)]
        FACTOR = {("kL", "L"): 1000, ("BX", "RD"): 20}               # (제출 단위, 품목 기준 단위) -> 환산 계수
        d["qty"] = d["qty"] * [FACTOR.get((u, UNIT_CODE[i]), 1) for u, i in zip(d["uom"], d["item"])]
        sign = {"issue": 1, "return": -1, "adjust": 1}
    else:
        d = d[d["item"].isin(UNIT_CODE) & (d["kind"] != "adjust")]
        sign = {"issue": 1, "return": -1}
    d = d.assign(v=d["kind"].map(sign) * d["qty"])
    return d.groupby(["unit", "item"])["v"].sum().round(6)

def compare(agg):
    tt = pd.Series(truth)
    tt.index.names = ["unit", "item"]
    diff = (agg.reindex(tt.index).fillna(0) - tt)
    return diff[diff.abs() > 0.5]

v1, v2 = compare(aggregate(S, "v1")), compare(aggregate(S, "v2"))
print(f"[AAR 오집계 분석] 제출 레코드 {len(S)}건, 정답 대조 대상 {len(truth)}개 (부대 x 품목) [교육용가상자료]")
print(f"룰셋 v1 오집계 {len(v1)}건(오차 합계 {v1.abs().sum():.0f}), 룰셋 v2 오집계 {len(v2)}건")
print("\n[사례별 원인 분석 - v1]")
rows = []
for name, ks in CASE.items():
    hit = [k for k in ks if k in v1.index]
    fix = [k for k in ks if k not in v2.index]
    err = float(sum(abs(v1[k]) for k in hit))
    rows.append({"사례": name, "발생 셀": len(ks), "v1 오류 셀": len(hit), "v1 오차": round(err), "v2 정상": f"{len(fix)}/{len(ks)}"})
print(pd.DataFrame(rows).to_string(index=False))
CAUSE = {"중복 행": "문서번호 중복 제거 규칙 없음", "단위 혼용(kL)": "단위 환산표 없음", "정정 누락": "정정(음수 조정) 문서를 집계에서 제외",
         "박스·발 혼용": "BX->RD 환산표 없음", "품목 별칭 누락": "별칭 사전 없고 미등록 품목을 조용히 버림"}
print("\n[원인 -> 룰셋 보정]")
for k, v in CAUSE.items():
    print(f" - {k}: {v} -> v2 패치: " + {"중복 행": "doc 기준 drop_duplicates", "단위 혼용(kL)": "kL x1000", "정정 누락": "adjust 포함 집계",
                                          "박스·발 혼용": "BX x20", "품목 별칭 누락": "별칭 사전 + 미등록 품목은 로그로 보류"}[k])

# 3) 회귀 시험 및 UAT 체크리스트(자동 점검 20항목)
agg2 = aggregate(S, "v2")
checks = [("정답 대조 오집계 0건", len(v2) == 0), ("중복 문서 제거", agg2.equals(aggregate(S.drop_duplicates("doc"), "v2"))),
          ("v2 재실행 결과 동일(멱등)", aggregate(S, "v2").equals(agg2)),
          ("kL 부대 4곳 정상", all(k not in v2.index for k in CASE["단위 혼용(kL)"])), ("BX 부대 4곳 정상", all(k not in v2.index for k in CASE["박스·발 혼용"])),
          ("정정 문서 5건 반영", all(k not in v2.index for k in CASE["정정 누락"])), ("별칭 5건 반영", all(k not in v2.index for k in CASE["품목 별칭 누락"])),
          ("정상 부대 v1 결과와 불변", len(set(truth) - set(v1.index) - set(v2.index)) == len(truth) - len(v1)),
          ("전체 불출 합계 정답과 일치", abs(agg2.sum() - sum(truth.values())) < 1), ("부대 24개 모두 집계", agg2.index.get_level_values(0).nunique() == 24),
          ("품목 11종 모두 집계", agg2.index.get_level_values(1).nunique() == 11), ("음수 순불출 0건", (agg2 >= 0).all()),
          ("미등록 품목 입력 시 오류 아닌 보류", aggregate(pd.concat([S, pd.DataFrame([{"doc": "X1", "unit": UNITS[0], "item": "미등록", "kind": "issue", "qty": 5, "uom": "EA"}])]), "v2").shape == agg2.shape),
          ("빈 입력에서도 오류 없음", aggregate(S.iloc[0:0], "v2").empty), ("처리 결과 부대x품목 단일 키", agg2.index.is_unique),
          ("v1 대비 오차 감소 100%", len(v2) == 0 and len(v1) > 0), ("단위 환산 후 소수 오차 없음", bool(((agg2 - agg2.round()).abs() < 1e-6).all())),
          ("사례 5유형 모두 재현·분석", len(CASE) == 5), ("원인-패치 대응표 작성", len(CAUSE) == 5), ("개선 요구 5건 반영", len(CASE) == 5 and len(v2) == 0)]
passed = sum(ok for _, ok in checks)
score = 100 * passed / len(checks)
print("\n[UAT 체크리스트]", f"{passed}/{len(checks)}항목 통과, 점수 {score:.0f}점")
for n, ok in checks:
    if not ok:
        print("  미통과:", n)
print("\n[개선 요구 반영 현황]")
for i, (k, ks) in enumerate(CASE.items(), 1):
    print(f" 요구 {i}. {k} 대응: {'반영 완료' if all(x not in v2.index for x in ks) else '미반영'}")
print(f"\n목표(UAT 90점 이상, 개선 요구 5건 반영): {'달성' if score >= 90 and len(v2) == 0 else '미달'}")
print("패치 내역: 룰셋 v1 -> v2 (중복 제거, 단위 환산표, 정정 반영, 별칭 사전, 미등록 품목 보류). 재집계 전 담당자가 보류 건을 확인한다.")


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 오집계 분석] 제출 레코드 539건, 정답 대조 대상 264개 (부대 x 품목) [교육용가상자료]
룰셋 v1 오집계 24건(오차 합계 1787), 룰셋 v2 오집계 0건

[사례별 원인 분석 - v1]
       사례  발생 셀  v1 오류 셀  v1 오차 v2 정상
     중복 행     6        6    546   6/6
단위 혼용(kL)     4        4    241   4/4
    정정 누락     5        5     71   5/5
  박스·발 혼용     4        4    367   4/4
 품목 별칭 누락     5        5    563   5/5

[원인 -> 룰셋 보정]
 - 중복 행: 문서번호 중복 제거 규칙 없음 -> v2 패치: doc 기준 drop_duplicates
 - 단위 혼용(kL): 단위 환산표 없음 -> v2 패치: kL x1000
 - 정정 누락: 정정(음수 조정) 문서를 집계에서 제외 -> v2 패치: adjust 포함 집계
 - 박스·발 혼용: BX->RD 환산표 없음 -> v2 패치: BX x20
 - 품목 별칭 누락: 별칭 사전 없고 미등록 품목을 조용히 버림 -> v2 패치: 별칭 사전 + 미등록 품목은 로그로 보류

[UAT 체크리스트] 20/20항목 통과, 점수 100점

[개선 요구 반영 현황]
 요구 1. 중복 행 대응: 반영 완료
 요구 2. 단위 혼용(kL) 대응: 반영 완료
 요구 3. 정정 누락 대응: 반영 완료
 요구 4. 박스·발 혼용 대응: 반영 완료
 요구 5. 품목 별칭 누락 대응: 반영 완료

목표(UAT 90점 이상, 개선 요구 5건 반영): 달성
패치 내역: 룰셋 v1 -> v2 (중복 제거, 단위 환산표, 정정 반영, 별칭 사전, 미등록 품목 보류). 재집계 전 담당자가 보류 건을 확인한다.
```
</details>
